# **Cuaderno exploratorio: priorizar la cobranza prediciendo la recuperación.**

**Qué es este cuaderno.** Es un análisis **aparte y exploratorio**, posterior al proyecto principal. No cambia su veredicto (SE REFORMULA). Prueba si la reformulación propuesta funciona: en lugar de estimar la LGD con la información de la originación, partir del saldo al default, que el banco ya conoce, y predecir solo **cuánto de ese saldo se recupera**.

**La idea.** La LGD se puede escribir separando lo que ya se sabe al default de lo que es incierto:

$$LGD = (1 - f)\times(1 - RR), \qquad f = \frac{\text{capital pagado}}{\text{monto}}, \qquad RR = \frac{\text{recuperación} - \text{comisión}}{\text{saldo al default}}$$

La pérdida esperada en dólares que se usa para ordenar los créditos es:

$$\text{Pérdida esperada}_i = \text{saldo al default}_i \times (1 - \widehat{RR}_i)$$

Si $\widehat{RR}$ fuera igual para todos, este orden sería exactamente el orden por saldo (40,6 %). El modelo solo puede ganar si **reordena lo que el saldo no distingue**: créditos con saldos parecidos pero recuperaciones distintas.

**Cómo está organizado.**

| Sección | Qué se hace |
|---|---|
| 1 | Carga y preparación (misma población, LGD, ventana de maduración y partición del proyecto principal) |
| 2 | Variables disponibles al momento del default, y exclusión de las que se conocen después |
| 3 | Descripción de la recuperación (cuántos créditos recuperan 0) |
| 4 | Modelo en dos partes con LightGBM, para ordenar |
| 5 | Regresión bayesiana de la recuperación (Gibbs), para la explicación y los intervalos |
| 6 | Evaluación en 2015: captura@20 % frente al orden por saldo, con intervalos |

## **1. Carga y preparación.**

Se reproduce exactamente la preparación del proyecto principal para que los créditos de prueba (2015) sean los mismos. Se cargan además cinco variables nuevas de la originación: FICO (rango bajo y alto), antigüedad laboral, tipo de vivienda y verificación del ingreso.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import kagglehub
import glob
import os
import lightgbm as lgb
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin

In [ ]:
semilla = 2026
rng = np.random.default_rng(semilla)
sns.set_theme(style='whitegrid', context='notebook', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 130, 'axes.titleweight': 'bold', 'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
traduccion = {
    'id': 'id_prestamo', 'loan_status': 'estado', 'funded_amnt': 'monto',
    'total_rec_prncp': 'capital_pagado', 'recoveries': 'recuperacion',
    'collection_recovery_fee': 'comision', 'annual_inc': 'ingreso', 'dti': 'dti',
    'grade': 'calificacion', 'term': 'plazo', 'purpose': 'proposito',
    'issue_d': 'fecha_originacion', 'last_pymnt_d': 'fecha_ultimo_pago',
    # variables nuevas de la originación
    'fico_range_low': 'fico_bajo', 'fico_range_high': 'fico_alto', 'emp_length': 'antiguedad_laboral',
    'home_ownership': 'vivienda', 'verification_status': 'verificacion',
}

In [ ]:
propositos = {
    'debt_consolidation': 'Consolidación de deudas', 'credit_card': 'Tarjeta de crédito',
    'home_improvement': 'Mejoras del hogar', 'major_purchase': 'Compra importante',
    'small_business': 'Pequeño negocio', 'car': 'Vehículo', 'medical': 'Gastos médicos',
    'moving': 'Mudanza', 'vacation': 'Vacaciones', 'house': 'Vivienda', 'wedding': 'Boda',
    'renewable_energy': 'Energía renovable', 'educational': 'Educación', 'other': 'Otro',
}

In [ ]:
# Fuente de datos: la copia versionada del proyecto en Google Drive (descarga original del 26 de octubre de 2026).
# Si no se encuentra, se descarga de Kaggle. La copia se crea una sola vez con Proyecto_LGD_VersionadoBase.ipynb.
RUTA_VERSIONADA = '/content/drive/MyDrive/LGD_Base_Versionada/lending_club_accepted_2007_2018Q4.parquet'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Sin Google Drive: se usará Kaggle si la copia versionada no está disponible')

In [ ]:
# Carga las columnas pedidas desde la copia versionada o, si no existe, desde Kaggle
def cargar_base(columnas):
    if os.path.exists(RUTA_VERSIONADA):
        print(f'Fuente: copia versionada ({RUTA_VERSIONADA})')
        return pd.read_parquet(RUTA_VERSIONADA, columns=columnas)
    print('Fuente: Kaggle (no se encontró la copia versionada)')
    carpeta = kagglehub.dataset_download('wordsforthewise/lending-club')
    candidatos = [a for a in glob.glob(os.path.join(carpeta, '**', 'accepted_2007_to_2018*.csv*'), recursive=True) if os.path.isfile(a)]
    ruta = sorted(candidatos, key=lambda a: not a.endswith('.gz'))[0]
    return pd.read_csv(ruta, usecols=columnas, dtype={'id': str}, low_memory=False)

In [ ]:
df = cargar_base(list(traduccion.keys()))
if df.shape[0] < 2_000_000:
    raise ValueError(f'Base incompleta: {df.shape[0]:,} filas')
df.shape

In [ ]:
# Filas de resumen (id no numérico), nombres en español y fechas
df = df[pd.to_numeric(df['id'], errors='coerce').notna()].reset_index(drop=True)
df = df.rename(columns=traduccion)
df['plazo'] = df['plazo'].str.strip().map({'36 months': '36 meses', '60 months': '60 meses'})
df['proposito'] = df['proposito'].map(propositos)
df['fecha_originacion'] = pd.to_datetime(df['fecha_originacion'], format='%b-%Y')
df['fecha_ultimo_pago'] = pd.to_datetime(df['fecha_ultimo_pago'], format='%b-%Y')

In [ ]:
# Población: créditos castigados (incluye fuera de política); se excluyen los 'Default'
df_def = df[df['estado'].str.contains('Charged Off', na=False)].copy()
df_def['lgd'] = ((df_def['monto'] - df_def['capital_pagado'] - (df_def['recuperacion'] - df_def['comision'])) / df_def['monto']).clip(0, 1)
len(df_def)

In [ ]:
# Ventana de maduración de 15 meses desde el default (mismo tratamiento de la censura del proyecto principal)
fecha_corte = df['fecha_ultimo_pago'].max()
df_def['fecha_default'] = df_def['fecha_ultimo_pago'].fillna(df_def['fecha_originacion'])
df_def['meses_desde_default'] = (fecha_corte.year - df_def['fecha_default'].dt.year) * 12 + (fecha_corte.month - df_def['fecha_default'].dt.month)
df_def['anio_originacion'] = df_def['fecha_originacion'].dt.year
df_modelo = df_def[df_def['meses_desde_default'] >= 15].reset_index(drop=True)
len(df_modelo)

In [ ]:
# Mismas exclusiones para el ajuste: ingreso <= 0, dti vacío o mayor que 100
df_ajuste = df_modelo[~((df_modelo['ingreso'] <= 0) | df_modelo['dti'].isna() | (df_modelo['dti'] > 100))].reset_index(drop=True)
len(df_ajuste)

## **2. Variables disponibles al momento del default.**

#### **Problema: usar información que solo se conoce después del default (fuga de información).**

La priorización se decide **cuando el crédito cae en default**, así que solo se pueden usar variables conocidas en ese momento. Se **excluyen** todas las que se registran después: `recoveries` y `collection_recovery_fee` (son el objetivo), `total_pymnt` (incluye las recuperaciones), `last_fico_range_*` (FICO actualizado después) y `out_prncp`.

| Variable | Qué es | ¿Se conoce al default? |
|---|---|---|
| Saldo al default | monto − capital pagado | Sí |
| Fracción pagada ($f$) | capital pagado / monto | Sí |
| Fracción del plazo cumplida | meses desde la originación hasta el default / plazo | Sí |
| Calificación, plazo, propósito, monto, ingreso, dti | Originación | Sí |
| FICO, antigüedad laboral, vivienda, verificación del ingreso | Originación | Sí |
| Recuperación, comisión, total pagado, FICO actualizado | Después del default | **No: se excluyen** |

In [ ]:
# Variables al momento del default
df_ajuste['saldo_default'] = df_ajuste['monto'] - df_ajuste['capital_pagado']
df_ajuste['fraccion_pagada'] = df_ajuste['capital_pagado'] / df_ajuste['monto']
plazo_meses = df_ajuste['plazo'].str.extract(r'(\d+)')[0].astype(float)
meses_hasta_default = (df_ajuste['fecha_default'].dt.year - df_ajuste['anio_originacion']) * 12 + (df_ajuste['fecha_default'].dt.month - df_ajuste['fecha_originacion'].dt.month)
df_ajuste['fraccion_plazo'] = (meses_hasta_default / plazo_meses).clip(0, 1.5)

In [ ]:
# Variables nuevas de la originación
df_ajuste['fico'] = (df_ajuste['fico_bajo'] + df_ajuste['fico_alto']) / 2
df_ajuste['antiguedad_laboral'] = df_ajuste['antiguedad_laboral'].fillna('Sin dato')
df_ajuste['vivienda'] = df_ajuste['vivienda'].where(df_ajuste['vivienda'].isin(['MORTGAGE', 'RENT', 'OWN']), 'OTRA')
df_ajuste['log2_ingreso'] = np.log2(df_ajuste['ingreso'])

In [ ]:
# Objetivo: tasa de recuperación del saldo (solo existe si hubo saldo al default)
df_ajuste['rr'] = ((df_ajuste['recuperacion'] - df_ajuste['comision']) / df_ajuste['saldo_default']).where(df_ajuste['saldo_default'] > 0).clip(0, 1)
df_ajuste[['saldo_default', 'fraccion_pagada', 'fraccion_plazo', 'fico', 'rr']].describe().round(3)

In [ ]:
# Partición temporal del proyecto principal: entrenamiento 2007-2014, prueba 2015 (se abre solo en la sección 6)
entrenamiento = df_ajuste[df_ajuste['anio_originacion'] <= 2014].reset_index(drop=True)
prueba = df_ajuste[df_ajuste['anio_originacion'] == 2015].reset_index(drop=True)
print(f'Entrenamiento: {len(entrenamiento):,} | Prueba: {len(prueba):,}')

## **3. ¿Cómo es la recuperación?**

#### **Problema: muchos créditos recuperan cero.**

Si una parte grande de los créditos no recupera nada y el resto recupera montos muy variables, un solo modelo lineal no distingue bien a unos de otros. Por eso se usa un **modelo en dos partes**:

$$\widehat{RR}_i = \underbrace{P(RR_i > 0)}_{\text{¿recupera algo?}} \times \underbrace{E[RR_i \mid RR_i > 0]}_{\text{¿cuánto, si recupera?}}$$

In [ ]:
con_saldo_ent = entrenamiento[entrenamiento['saldo_default'] > 0]
print(f"Créditos de entrenamiento con saldo: {len(con_saldo_ent):,}")
print(f"Recuperan 0: {(con_saldo_ent['rr'] == 0).mean() * 100:.1f} % | Recuperación promedio: {con_saldo_ent['rr'].mean() * 100:.1f} % | Promedio cuando recuperan algo: {con_saldo_ent.loc[con_saldo_ent['rr'] > 0, 'rr'].mean() * 100:.1f} %")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(con_saldo_ent['rr'], bins=50, color='#1f4e79', ax=ax)
ax.set_title('¿Qué parte del saldo se recupera después del castigo?')
ax.set_xlabel('Tasa de recuperación del saldo')
ax.set_ylabel('Número de créditos')
plt.show()

In [ ]:
# Recuperación por calificación: ¿la calificación distingue quién recupera más?
con_saldo_ent.groupby('calificacion')['rr'].agg(creditos='size', recupera_algo=lambda s: (s > 0).mean() * 100, rr_promedio=lambda s: s.mean() * 100).round(1)

## **4. Modelo en dos partes con LightGBM (para ordenar).**

Para **ordenar** no hace falta un modelo calibrado ni lineal: basta con que ponga primero a los créditos que menos van a recuperar. LightGBM (árboles de decisión potenciados) capta relaciones no lineales e interacciones sin especificarlas a mano. Se usan dos modelos:

1. **Clasificador:** probabilidad de que el crédito recupere algo ($RR > 0$).
2. **Regresor:** cuánto recupera, entrenado solo con los créditos que sí recuperan.

Los hiperparámetros se fijan antes de ver la prueba y no se ajustan con ella. La validación cruzada usa los mismos años del proyecto principal (2012, 2013 y 2014).

In [ ]:
numericas = ['saldo_default', 'fraccion_pagada', 'fraccion_plazo', 'monto', 'log2_ingreso', 'dti', 'fico']
categoricas = ['calificacion', 'plazo', 'proposito', 'antiguedad_laboral', 'vivienda', 'verificacion']
predictoras = numericas + categoricas

In [ ]:
def matriz_lgb(datos):
    x = datos[predictoras].copy()
    for c in categoricas:
        x[c] = x[c].astype('category')
    return x

In [ ]:
parametros_lgb = dict(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=200,
                      subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=semilla, verbose=-1)

def ajustar_dos_partes(datos):
    datos = datos[datos['saldo_default'] > 0]
    clasificador = lgb.LGBMClassifier(**parametros_lgb).fit(matriz_lgb(datos), (datos['rr'] > 0).astype(int))
    recuperan = datos[datos['rr'] > 0]
    regresor = lgb.LGBMRegressor(**parametros_lgb).fit(matriz_lgb(recuperan), recuperan['rr'])
    return clasificador, regresor

def predecir_rr(modelos, datos):
    clasificador, regresor = modelos
    x = matriz_lgb(datos)
    return np.clip(clasificador.predict_proba(x)[:, 1] * regresor.predict(x), 0, 1)

In [ ]:
def captura(perdida_real, orden, proporcion=0.20):
    n_top = int(len(orden) * proporcion)
    top = np.argsort(-np.asarray(orden), kind='stable')[:n_top]
    return np.asarray(perdida_real)[top].sum() / np.asarray(perdida_real).sum() * 100

In [ ]:
# Validación cruzada temporal (ventana expansiva): captura del orden por saldo frente al orden con RR predicha
filas_cv = []
for ultimo, anio_val in [(2011, 2012), (2012, 2013), (2013, 2014)]:
    ajuste = entrenamiento[entrenamiento['anio_originacion'] <= ultimo]
    validacion = entrenamiento[entrenamiento['anio_originacion'] == anio_val]
    rr_hat = predecir_rr(ajustar_dos_partes(ajuste), validacion)
    perdida_real = validacion['lgd'] * validacion['monto']
    filas_cv.append({'Valida con': anio_val,
                     'Captura orden por saldo (%)': captura(perdida_real, validacion['saldo_default']),
                     'Captura saldo x (1 - RR predicha) (%)': captura(perdida_real, validacion['saldo_default'] * (1 - rr_hat)),
                     'Máximo posible (%)': captura(perdida_real, perdida_real)})
tabla_cv = pd.DataFrame(filas_cv)
tabla_cv['Diferencia (pp)'] = tabla_cv['Captura saldo x (1 - RR predicha) (%)'] - tabla_cv['Captura orden por saldo (%)']
tabla_cv.round(2)

In [ ]:
# Ajuste final con todo el entrenamiento
modelos_lgb = ajustar_dos_partes(entrenamiento)

In [ ]:
# Importancia de las variables en el clasificador (¿recupera algo?)
pd.Series(modelos_lgb[0].booster_.feature_importance(importance_type='gain'), index=predictoras).sort_values(ascending=False).round(0)

## **5. Regresión bayesiana de la recuperación (explicación e intervalos).**

LightGBM ordena, pero no entrega coeficientes interpretables ni intervalos de credibilidad. Para explicar **qué se asocia con recuperar más** se ajusta la misma regresión lineal bayesiana del proyecto principal (priors semiconjugadas y Gibbs), ahora sobre $RR$ y con las predictoras al default:

$$RR_i \mid \beta, \sigma^2 \sim \text{Normal}(x_i^\top\beta,\ \sigma^2), \qquad \beta_j \sim \text{Normal}(\beta_{0j}, \tau_j^2), \qquad \sigma^2 \sim \text{Gamma-Inversa}\left(\tfrac{\nu_0}{2}, \tfrac{\nu_0\sigma_0^2}{2}\right)$$

**Priors.** Intercepto: media 0,5 y desviación 0,25, prácticamente no informativa dentro de [0, 1] (no se supone cuánto se recupera). Efectos: media 0 (sin dirección supuesta) y desviación 0,10 en las categorías y en las fracciones, y más angosta en las variables con unidades grandes. $\sigma^2$: $\nu_0 = 5$, $\sigma_0 = 0{,}25$. Se usan 4 cadenas de 6.000 iteraciones, quemando las primeras 1.000.

In [ ]:
referencias = {'calificacion': 'A', 'plazo': '36 meses', 'proposito': 'Consolidación de deudas',
               'antiguedad_laboral': '10+ years', 'vivienda': 'MORTGAGE', 'verificacion': 'Not Verified'}

In [ ]:
class PreparacionRR(BaseEstimator, TransformerMixin):
    # Centra las numéricas con las medias del entrenamiento y crea las dummies a mano (referencias sin columna)
    def fit(self, X, y=None):
        self.medias_ = X[numericas].mean()
        self.categorias_ = {v: sorted(X[v].unique()) for v in referencias}
        return self

    def transform(self, X):
        salida = pd.DataFrame(index=X.index)
        salida['intercepto'] = 1.0
        salida['saldo_miles_c'] = (X['saldo_default'] - self.medias_['saldo_default']) / 1000
        salida['fraccion_pagada_c'] = X['fraccion_pagada'] - self.medias_['fraccion_pagada']
        salida['fraccion_plazo_c'] = X['fraccion_plazo'] - self.medias_['fraccion_plazo']
        salida['monto_miles_c'] = (X['monto'] - self.medias_['monto']) / 1000
        salida['log2_ingreso_c'] = X['log2_ingreso'] - self.medias_['log2_ingreso']
        salida['dti_c'] = X['dti'] - self.medias_['dti']
        salida['fico_c'] = (X['fico'] - self.medias_['fico']) / 10
        for variable, referencia in referencias.items():
            for categoria in self.categorias_[variable]:
                if categoria != referencia:
                    salida[f'{variable}_{categoria}'] = (X[variable] == categoria).astype(float)
        return salida

In [ ]:
# Priors: (media, desviación) por grupo de coeficientes
prior_grupos = {'intercepto': (0.5, 0.25), 'saldo_miles_c': (0.0, 0.01), 'monto_miles_c': (0.0, 0.01),
                'fraccion_pagada_c': (0.0, 0.10), 'fraccion_plazo_c': (0.0, 0.10), 'log2_ingreso_c': (0.0, 0.05),
                'dti_c': (0.0, 0.005), 'fico_c': (0.0, 0.02), 'calificacion_': (0.0, 0.10), 'plazo_': (0.0, 0.10),
                'proposito_': (0.0, 0.10), 'antiguedad_laboral_': (0.0, 0.10), 'vivienda_': (0.0, 0.10), 'verificacion_': (0.0, 0.10)}
nu0, s02 = 5, 0.25 ** 2

def vectores_prior(nombres_columnas):
    grupos = [next(g for g in prior_grupos if n == g or n.startswith(g)) for n in nombres_columnas]
    return np.array([prior_grupos[g][0] for g in grupos]), np.array([prior_grupos[g][1] for g in grupos])

In [ ]:
# Muestreador de Gibbs (mismas condicionales completas del proyecto principal)
def gibbs(x, y, beta0, sd_prior, n_iter, semilla_cadena, beta_inicial, sigma2_inicial):
    generador = np.random.default_rng(semilla_cadena)
    n, p = x.shape
    xtx, xty, yty = x.T @ x, x.T @ y, y @ y
    precision0 = np.diag(1 / sd_prior ** 2)
    beta, sigma2 = beta_inicial.copy(), sigma2_inicial
    muestras_beta, muestras_sigma2 = np.empty((n_iter, p)), np.empty(n_iter)
    for t in range(n_iter):
        v_n = np.linalg.inv(precision0 + xtx / sigma2)
        beta = v_n @ (precision0 @ beta0 + xty / sigma2) + np.linalg.cholesky(v_n) @ generador.standard_normal(p)
        scr = yty - 2 * beta @ xty + beta @ xtx @ beta
        sigma2 = 1 / generador.gamma((nu0 + n) / 2, 1 / ((nu0 * s02 + scr) / 2))
        muestras_beta[t], muestras_sigma2[t] = beta, sigma2
    return muestras_beta, muestras_sigma2

In [ ]:
# Ajuste bayesiano: 4 cadenas, 6.000 iteraciones, se queman las primeras 1.000
preparacion = PreparacionRR().fit(con_saldo_ent)
x_bayes = preparacion.transform(con_saldo_ent)
nombres = list(x_bayes.columns)
beta0, sd_prior = vectores_prior(nombres)
prior_s2 = stats.invgamma(nu0 / 2, scale=nu0 * s02 / 2)
cadenas = [gibbs(x_bayes.to_numpy(), con_saldo_ent['rr'].to_numpy(), beta0, sd_prior, 6000, semilla + 100 + k,
                 np.random.default_rng(semilla + k).normal(beta0, sd_prior), prior_s2.rvs(random_state=semilla + k))
           for k in range(4)]
post_beta = np.array([c[0] for c in cadenas])[:, 1000:, :]
muestras = post_beta.reshape(-1, len(nombres))

In [ ]:
# Convergencia: R-hat dividido (Gelman-Rubin)
def r_hat(c):
    m, n = c.shape
    n2 = n // 2
    mitades = c[:, :n2 * 2].reshape(m * 2, n2)
    w = mitades.var(axis=1, ddof=1).mean()
    b = n2 * mitades.mean(axis=1).var(ddof=1)
    return np.sqrt(((n2 - 1) / n2 * w + b / n2) / w)

print(f'R-hat máximo: {max(r_hat(post_beta[:, :, j]) for j in range(len(nombres))):.4f}')

In [ ]:
# Qué se asocia con recuperar más: puntos porcentuales de recuperación del saldo, con intervalo de credibilidad del 95 %
tabla_rr = pd.DataFrame({'Media posterior (pp)': muestras.mean(axis=0) * 100,
                         'Límite inferior 95 % (pp)': np.percentile(muestras, 2.5, axis=0) * 100,
                         'Límite superior 95 % (pp)': np.percentile(muestras, 97.5, axis=0) * 100,
                         'P(efecto > 0)': (muestras > 0).mean(axis=0)}, index=nombres)
tabla_rr.round(2)

## **6. Evaluación en la prueba de 2015.**

Se abre la base de prueba por única vez. Se comparan, sobre los mismos créditos de 2015 del proyecto principal:

| Orden | Pérdida esperada usada para ordenar |
|---|---|
| Por saldo al default (priorización actual) | $\text{saldo}_i$ |
| Saldo × (1 − RR predicha por LightGBM) | $\text{saldo}_i \times (1 - \widehat{RR}^{\,LGB}_i)$ |
| Saldo × (1 − RR predicha por el modelo bayesiano) | $\text{saldo}_i \times (1 - \widehat{RR}^{\,bayes}_i)$ |
| Máximo posible | Pérdida real |

**Cuándo diríamos que funciona:** si el orden con la recuperación predicha supera al orden por saldo con un intervalo bootstrap que no incluya el cero, y si llega al 40,9 % fijado en el proyecto principal.

In [ ]:
perdida_real_p = (prueba['lgd'] * prueba['monto']).to_numpy()
saldo_p = prueba['saldo_default'].to_numpy()
rr_lgb_p = predecir_rr(modelos_lgb, prueba)
rr_bayes_p = np.clip(preparacion.transform(prueba).to_numpy() @ muestras.mean(axis=0), 0, 1)

In [ ]:
capturas = pd.Series({
    'Orden por saldo al default (priorización actual)': captura(perdida_real_p, saldo_p),
    'Saldo x (1 - RR predicha por LightGBM)': captura(perdida_real_p, saldo_p * (1 - rr_lgb_p)),
    'Saldo x (1 - RR predicha bayesiana)': captura(perdida_real_p, saldo_p * (1 - rr_bayes_p)),
    'Máximo posible (pérdida real)': captura(perdida_real_p, perdida_real_p),
}, name='Captura@20 % (%)')
capturas.round(2)

In [ ]:
# Bootstrap de 2015: diferencia de captura de cada orden frente al saldo, en las mismas réplicas
n_top = int(len(prueba) * 0.20)
def captura_rapida(perdida, orden):
    return perdida[np.argpartition(-orden, n_top)[:n_top]].sum() / perdida.sum() * 100

dif_lgb, dif_bayes = np.empty(1000), np.empty(1000)
for b in range(1000):
    i = rng.integers(0, len(prueba), len(prueba))
    base = captura_rapida(perdida_real_p[i], saldo_p[i])
    dif_lgb[b] = captura_rapida(perdida_real_p[i], saldo_p[i] * (1 - rr_lgb_p[i])) - base
    dif_bayes[b] = captura_rapida(perdida_real_p[i], saldo_p[i] * (1 - rr_bayes_p[i])) - base
for nombre, d in [('LightGBM', dif_lgb), ('Bayesiano', dif_bayes)]:
    print(f'{nombre} frente al saldo: {d.mean():+.2f} pp | IC 95 %: [{np.percentile(d, 2.5):+.2f}; {np.percentile(d, 97.5):+.2f}] | P(mejor que el saldo) = {(d > 0).mean():.3f}')

In [ ]:
# Precisión de la LGD reconstruida (1 - f) x (1 - RR predicha), frente a la línea base justa (1 - f) x (1 - RR promedio)
f_p = prueba['fraccion_pagada'].to_numpy()
y_p = prueba['lgd'].to_numpy()
rmse = lambda a, b: np.sqrt(np.mean((a - b) ** 2))
rmse_justa = rmse(y_p, (1 - f_p) * (1 - con_saldo_ent['rr'].mean()))
for nombre, rr_hat in [('LightGBM', rr_lgb_p), ('Bayesiano', rr_bayes_p)]:
    r = rmse(y_p, (1 - f_p) * (1 - rr_hat))
    print(f'{nombre}: RMSE {r:.4f} | línea base justa {rmse_justa:.4f} | mejora {(1 - r / rmse_justa) * 100:.1f} %')

In [ ]:
# Curva de ganancia acumulada
fig, ax = plt.subplots(figsize=(7, 6))
for orden, nombre, estilo in [(perdida_real_p, 'Máximo posible', ':'), (saldo_p * (1 - rr_lgb_p), 'Saldo x (1 - RR LightGBM)', '-'),
                              (saldo_p, 'Saldo al default (actual)', '--')]:
    idx = np.argsort(-orden, kind='stable')
    ax.plot(np.arange(1, len(idx) + 1) / len(idx) * 100, np.cumsum(perdida_real_p[idx]) / perdida_real_p.sum() * 100, estilo, label=nombre)
ax.plot([0, 100], [0, 100], color='gray', linewidth=0.8, label='Orden al azar')
ax.axvline(20, color='#c0392b', linewidth=0.8)
ax.set_title('¿Cuánta pérdida se alcanza a gestionar según el orden?')
ax.set_xlabel('% de créditos gestionados')
ax.set_ylabel('% de la pérdida total en USD')
ax.legend(frameon=False)
plt.show()

In [ ]:
# Resumen final del cuaderno exploratorio
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame({
        'Captura@20 % (%)': [f"{capturas.iloc[0]:.2f}", f"{capturas.iloc[1]:.2f}", f"{capturas.iloc[2]:.2f}", f"{capturas.iloc[3]:.2f}"],
        'Diferencia con el saldo [IC 95 % bootstrap]': ['—',
            f'{dif_lgb.mean():+.2f} pp [{np.percentile(dif_lgb, 2.5):+.2f}; {np.percentile(dif_lgb, 97.5):+.2f}]',
            f'{dif_bayes.mean():+.2f} pp [{np.percentile(dif_bayes, 2.5):+.2f}; {np.percentile(dif_bayes, 97.5):+.2f}]', '—'],
        '¿Alcanza el 40,9 %?': ['No' if capturas.iloc[0] < 40.9 else 'Sí', 'No' if capturas.iloc[1] < 40.9 else 'Sí',
                                'No' if capturas.iloc[2] < 40.9 else 'Sí', 'Techo'],
    }, index=capturas.index))